In [ ]:
---
toc: false 
layout: post
title: Fibonacci Jumper 2
description: An AP CSA game teaching arrays, loops and classes through Fibonacci.
courses: { csa: {week: 25} }
type: ccc
image: /images/data_structures/fibonacci.png
permalink: /fibonacci2
---


![game](https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Fibonacci_Squares.svg/1200px-Fibonacci_Squares.svg.png)

## Fibonacci Jumper Game

Welcome to **Fibonacci Jumper**! A game where you control a character, and each jump must be the next Fibonacci number.

### AP CSA Concepts Taught:
- **Arrays/ArrayLists**: Tracking our Fibonacci sequence and jumps.
- **Loops**: Generating the jumps up to N limits.
- **Conditionals**: Win/Lose condition logic based on overshoot.
- **Methods**: Moving the character incrementally.
- **Classes**: Structuring the Game and Player logic.


In [ ]:
%%js

// GAME_RUNNER: Fibonacci Rift Rescue. | hide_edit: true

import { GameControl, GameEnvBackground, Player, NPC } from '/assets/js/GameEnginev1.1/essentials/Imports.js';
import Barrier from '/assets/js/GameEnginev1.1/essentials/Barrier.js';

class JumperPlayer extends Player {
  constructor(data, gameEnv) {
    super(data, gameEnv);

    this.fiboSeq = [1, 1, 2, 3, 5, 8, 13, 21, 34];
    this.jumpIndex = 0;

    this.energy = 0;
    this.targetEnergy = 33;
    this.safeMin = -8;
    this.safeMax = this.targetEnergy + 12;

    this.startX = this.position.x;
    this.scalar = gameEnv.innerWidth * 0.02;

    this.done = false;
    this._lastKeyDown = null;
  }

  updateVelocity() {
    // Disable continuous movement; movement only happens through Fibonacci jumps
    this.velocity.x = 0;
    this.velocity.y = 0;
  }

  resetRun() {
    this.jumpIndex = 0;
    this.energy = 0;
    this.done = false;
    this.position.x = this.startX;
    console.log('🔄 Rift reset. Begin the Fibonacci sequence again.');
  }

  handleKeyDown({ keyCode }) {
    if (this._lastKeyDown === keyCode) return;
    this._lastKeyDown = keyCode;

    if (keyCode === 82) {
      this.resetRun();
      return;
    }

    if (this.done) return;
    if (this.jumpIndex >= this.fiboSeq.length) {
      console.log('⚠️ No jumps remaining. Press R to restart.');
      return;
    }

    let dir = 0;
    if (keyCode === this.keypress.right) dir = 1;
    if (keyCode === this.keypress.left) dir = -1;
    if (dir === 0) return;

    const jumpVal = this.fiboSeq[this.jumpIndex];
    this.jumpIndex++;

    this.energy += dir * jumpVal;
    this.position.x = this.startX + (this.energy * this.scalar);

    if (this.energy === this.targetEnergy) {
      console.log(`✨ PERFECT RESONANCE! The Rift Core is charged to exactly ${this.targetEnergy}. You win!`);
      this.done = true;
      return;
    }

    if (this.energy > this.safeMax || this.energy < this.safeMin) {
      console.log('💀 The void consumed your path. You drifted out of the safe energy range.');
      this.done = true;
      return;
    }

    if (this.jumpIndex >= this.fiboSeq.length && this.energy !== this.targetEnergy) {
      console.log(`⚠️ Sequence exhausted. Final energy: ${this.energy}. Press R to try again.`);
      this.done = true;
      return;
    }

    console.log(
      `Jump ${dir > 0 ? 'RIGHT' : 'LEFT'} by ${jumpVal}. ` +
      `Current energy: ${this.energy}. ` +
      `Next Fibonacci jump: ${this.fiboSeq[this.jumpIndex] ?? 'none'}`
    );
  }

  handleKeyUp({ keyCode }) {
    if (this._lastKeyDown === keyCode) this._lastKeyDown = null;
  }
}

class FibonacciLevel {
  constructor(gameEnv) {
    const path = gameEnv.path;
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;

    const bgData = {
      name: 'fibonacci_bg',
      src: path + '/images/gamebuilder/bg/alien_planet.jpg',
      pixels: { height: 720, width: 1280 }
    };

    const playerData = {
      id: 'RiftRunner',
      src: path + '/images/gamify/chillguy.png',
      SCALE_FACTOR: 5,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: { x: width * 0.05, y: height * 0.72 },
      pixels: { height: 512, width: 384 },
      orientation: { rows: 4, columns: 3 },
      down: { row: 0, start: 0, columns: 3 },
      downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI / 16 },
      downLeft: { row: 2, start: 0, columns: 3, rotate: -Math.PI / 16 },
      right: { row: 1, start: 0, columns: 3 },
      left: { row: 2, start: 0, columns: 3 },
      up: { row: 3, start: 0, columns: 3 },
      upRight: { row: 1, start: 0, columns: 3, rotate: -Math.PI / 16 },
      upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI / 16 },
      hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
      keypress: { up: 87, left: 65, down: 83, right: 68 }
    };

    function station(id, greeting, xPos, yPos, scale) {
      return {
        id,
        greeting,
        SCALE_FACTOR: scale,
        visible: false,
        INIT_POSITION: { x: xPos, y: yPos },
        hitbox: { widthPercentage: 0, heightPercentage: 0 },
        reaction: function() {
          if (this.dialogueSystem) {
            this.showReactionDialogue();
          } else {
            console.log(this.greeting);
          }
        },
        interact: function() {
          if (this.dialogueSystem) {
            this.showRandomDialogue();
          } else {
            console.log(this.greeting);
          }
        }
      };
    }

    const stationRules = station(
      'Navigator Beacon',
      'Use A and D to move left or right. Each move must use the NEXT Fibonacci jump: 1, 1, 2, 3, 5, 8, 13...',
      width * 0.15, height * 0.30, 5
    );

    const stationTarget = station(
      'Core Terminal',
      'Mission: charge the Rift Core to exactly 33 energy. Precision matters.',
      width * 0.40, height * 0.55, 6
    );

    const stationBounds = station(
      'Void Warning',
      'Stray too far into negative or overflow energy, and the void collapses your route.',
      width * 0.65, height * 0.75, 7
    );

    const stationTip = station(
      'Oracle Node',
      'Early choices shape the whole run. Fibonacci jumps grow fast, so one mistake can echo forward.',
      width * 0.82, height * 0.45, 8
    );

    const stationReset = station(
      'Reset Shrine',
      'Press R at any time to restart the rift sequence.',
      width * 0.55, height * 0.28, 5
    );

    const topBarrier = {
      id: 'top_wall',
      xPercentage: 0.5,
      yPercentage: 0.1,
      widthPercentage: 1.0,
      heightPercentage: 0.03
    };

    const bottomBarrier = {
      id: 'bottom_wall',
      xPercentage: 0.5,
      yPercentage: 0.95,
      widthPercentage: 1.0,
      heightPercentage: 0.03
    };

    this.classes = [
      { class: GameEnvBackground, data: bgData },
      { class: Barrier, data: topBarrier },
      { class: Barrier, data: bottomBarrier },
      { class: JumperPlayer, data: playerData },
      { class: NPC, data: stationRules },
      { class: NPC, data: stationTarget },
      { class: NPC, data: stationBounds },
      { class: NPC, data: stationTip },
      { class: NPC, data: stationReset }
    ];
  }
}

export const gameLevelClasses = [FibonacciLevel];
export { GameControl };

In [ ]:
/*
 * Creator: Open Coding Society
 * Mini Lab Name: Fibonacci Jumper - Java Backend
 */

import java.util.ArrayList;

/* AP CSA Topic: Abstract Classes and Methods */
abstract class Game {
    String name;
    int currentPos;
    int targetPos;
    ArrayList<Integer> sequence; // AP CSA Topic: ArrayList

    public Game(String name, int targetPos) {
        this.name = name;
        this.targetPos = targetPos;
        this.currentPos = 0;
        this.sequence = new ArrayList<>();
    }

    // Abstract method to generate the sequence (AP CSA Topic: Arrays & Loops)
    protected abstract void generateSequence(int limit);

    // Method to jump (AP CSA Topic: Conditionals & Methods)
    public abstract boolean jump(int direction);
}


In [2]:
/* AP CSA Topic: Inheritance and Classes */
public class FibonacciJumper extends Game {
    private int jumpCount;

    public FibonacciJumper(int target) {
        super("Fibonacci Jumper", target);
        this.jumpCount = 0;
        generateSequence(15); // Pre-generate 15 potential jumps
    }

    @Override
    protected void generateSequence(int limit) {
        // AP CSA Topic: Iteration & Algorithms
        sequence.add(1);
        sequence.add(1);
        for(int i = 2; i < limit; i++) {
            sequence.add(sequence.get(i-1) + sequence.get(i-2));
        }
    }

    @Override
    public boolean jump(int direction) {
        // AP CSA Topic: Conditionals (if/else)
        if (jumpCount >= sequence.size()) {
            System.out.println("Out of jumps!");
            return false;
        }
        
        int jumpDist = sequence.get(jumpCount);
        if (direction > 0) {
            currentPos += jumpDist;
            System.out.println("Jumped RIGHT by " + jumpDist + ". Position: " + currentPos);
        } else if (direction < 0) {
            currentPos -= jumpDist;
            System.out.println("Jumped LEFT by " + jumpDist + ". Position: " + currentPos);
        }
        jumpCount++;

        return checkCondition();
    }

    private boolean checkCondition() {
        if (currentPos == targetPos) {
            System.out.println("🎉 YOU WIN! Landed exactly on " + targetPos + "!");
            return true;
        } else if (currentPos > targetPos + 20 || currentPos < -5) {
            System.out.println("💀 GAME OVER! Overshot the target into oblivion.");
            return true;
        }
        return false;
    }
}


Calculation method = FiboFor extends Fibo
fibonacci Number 2 = 1
fibonacci List = [0, 1]
fibonacci Hashmap = {0=[0], 1=[0, 1]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]

Calculation method = FiboFor extends Fibo
fibonacci Number 5 = 3
fibonacci List = [0, 1, 1, 2, 3]
fibonacci Hashmap = {0=[0], 1=[0, 1], 2=[0, 1, 1], 3=[0, 1, 1, 2], 4=[0, 1, 1, 2, 3]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]
fibonacci Sequence 3 = [0, 1, 1]
fibonacci Sequence 4 = [0, 1, 1, 2]
fibonacci Sequence 5 = [0, 1, 1, 2, 3]

Calculation method = FiboFor extends Fibo
fibonacci Number 8 = 13
fibonacci List = [0, 1, 1, 2, 3, 5, 8, 13]
fibonacci Hashmap = {0=[0], 1=[0, 1], 2=[0, 1, 1], 3=[0, 1, 1, 2], 4=[0, 1, 1, 2, 3], 5=[0, 1, 1, 2, 3, 5], 6=[0, 1, 1, 2, 3, 5, 8], 7=[0, 1, 1, 2, 3, 5, 8, 13]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]
fibonacci Sequence 3 = [0, 1, 1]
fibonacci Sequence 4 = [0, 1, 1, 2]
fibonacci Sequence 5 = [0, 1, 1, 2, 3]
fibonacci Sequence 6 = [0, 1,

In [3]:
/* AP CSA Topic: Main execution and Traced runs */
public class Tester {
    public static void main(String[] args) {
        // Target is 33
        // Optimal Jumps: 1(R)+1(R)+2(R)+3(R)+5(R)+8(R)+13(R) = 33
        FibonacciJumper game = new FibonacciJumper(33);
        
        System.out.println("Starting " + game.name + ". Target is " + game.targetPos);
        
        game.jump(1);  // +1
        game.jump(1);  // +1
        game.jump(1);  // +2
        game.jump(1);  // +3
        game.jump(1);  // +5
        game.jump(-1); // OH NO! Went left -8. Position should be 12 - 8 = 4
        game.jump(1);  // +13. Position 4 + 13 = 17.
        game.jump(1);  // +21. Position 17 + 21 = 38. OVER SHOT! (Target is 33)
    }
}
Tester.main(null);


Calculation method = FiboFor extends Fibo
fibonacci Number 2 = 1
fibonacci List = [0, 1]
fibonacci Hashmap = {0=[0], 1=[0, 1]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]

Calculation method = FiboFor extends Fibo
fibonacci Number 5 = 3
fibonacci List = [0, 1, 1, 2, 3]
fibonacci Hashmap = {0=[0], 1=[0, 1], 2=[0, 1, 1], 3=[0, 1, 1, 2], 4=[0, 1, 1, 2, 3]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]
fibonacci Sequence 3 = [0, 1, 1]
fibonacci Sequence 4 = [0, 1, 1, 2]
fibonacci Sequence 5 = [0, 1, 1, 2, 3]

Calculation method = FiboFor extends Fibo
fibonacci Number 8 = 13
fibonacci List = [0, 1, 1, 2, 3, 5, 8, 13]
fibonacci Hashmap = {0=[0], 1=[0, 1], 2=[0, 1, 1], 3=[0, 1, 1, 2], 4=[0, 1, 1, 2, 3], 5=[0, 1, 1, 2, 3, 5], 6=[0, 1, 1, 2, 3, 5, 8], 7=[0, 1, 1, 2, 3, 5, 8, 13]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]
fibonacci Sequence 3 = [0, 1, 1]
fibonacci Sequence 4 = [0, 1, 1, 2]
fibonacci Sequence 5 = [0, 1, 1, 2, 3]
fibonacci Sequence 6 = [0, 1,

## Popcorn Hacks
Objectives of these hacks are ...

1. Understand how to fullfill abstract class requirements using two additional algoritms.
2. Use inheritance style of programming to test speed of each algorithm.  To test the speed, a.) be aware that the first run is always the slowest b.) to time something, my recommendation is 12 runs on the timed element, through out highest and lowest time in calculations.
3. Be sure to make a tester and reporting methods.

.85 basis for text based comparison inside of Jupyter Notebook lesson

## Hacks
Assign in each Team to build a Thymeleaf UI for pages using this example https://thymeleaf.nighthawkcodingsociety.com/mvc/fibonacci as basis.  Encorporate into Algorithms menu.

Since there are three teams, one team can do Fibo, others Pali and Factorial.  Assign this to people that are struggling for contribution and presentation to checkpoints.

.90 basis for FE presentation in Thymmeleaf to BE call in Spring